# Proyecto Integrador — Persona C
**Tareas 4, 16, 17, 18, 19** · SEACE/OCDS 2024-01 a 2026-08 (OECE vía Open Contracting, CC BY 4.0)

| Bloque | Tarea | Dónde corre |
|---|---|---|
| 0. Setup | lectura de Parquet | Spark local |
| 1. Agregaciones RDD vs DataFrame + particionamiento | 4 | Spark |
| 2. Transacciones (canastas) | 16 | Spark |
| 3. Apriori (muestra) | 17 | Python local (numpy) sobre muestra |
| 4. FP-Growth (dataset completo) | 18 | Spark MLlib |
| 5. Reglas: soporte, confianza, lift + interpretación | 19 | Python (los itemsets frecuentes son pocos) |
| 6. Export de artefactos para la web | — | CSV/JSON |

**Preguntas de interés (Parte V):** ¿qué combinaciones de categoría, procedimiento, tipo de entidad, departamento, rubro (UNSPSC), monto y nº de postores se asocian a que un proceso termine DESIERTO o NULO? ¿Cambia el patrón antes/después de la Ley 32069 (22-abr-2025)?

In [1]:
# ── Solo Colab ──
!pip install -q pyspark==3.5.3
from google.colab import drive
drive.mount("/content/drive")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.3/317.3 MB 1.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 5.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.1.0 requires pyspark[connect]~=4.0.0, but you have pyspark 3.5.3 which is incompatible.
Mounted at /content/drive


In [2]:

# ────────────────
import time, json, os, itertools
import numpy as np, pandas as pd
from pyspark.sql import SparkSession, functions as F
spark = SparkSession.builder.master("local[*]").config("spark.driver.memory","8g").getOrCreate()
# NO cambiar spark.sql.session.timeZone (fechas ya en hora de Lima)

RUTA = "/content/drive/MyDrive/Compartidos/Proyecto Data Mining/proyecto_integrador_seace/parquet"          # <- ajusta
OUT  = "/content/drive/MyDrive/Compartidos/Proyecto Data Mining/artefactos_C"     # carpeta de salida para la web
os.makedirs(OUT, exist_ok=True)

procesos       = spark.read.parquet(f"{RUTA}/procesos")
items          = spark.read.parquet(f"{RUTA}/items")
adjudicaciones = spark.read.parquet(f"{RUTA}/adjudicaciones")
contratos      = spark.read.parquet(f"{RUTA}/contratos")
entidades      = spark.read.parquet(f"{RUTA}/entidades")
print(procesos.count(), items.count())

def cronometrar(f, n=3):
    ts=[]
    for _ in range(n):
        t=time.perf_counter(); r=f(); ts.append(time.perf_counter()-t)
    return r, min(ts), float(np.median(ts))

209740 243699


## 1. Tarea 4 — Agregaciones: RDD (MapReduce) vs DataFrame/Spark SQL
Dos agregaciones ligadas a las preguntas:
- **Agg 1:** nº de procesos, monto total (`valor_pen_valido`) y monto promedio por `entidad_departamento`.
- **Agg 2:** tasa de DESIERTO por `procedimiento` × `categoria` (excluyendo `CONVOCADO`, como indica el LEEME).

In [3]:
# ---------- Agg 1 ----------
base = procesos.select("entidad_departamento","categoria","procedimiento","estado_proceso","valor_pen_valido","anio_pub").cache()
base.count()

def agg1_df():
    return (base.groupBy("entidad_departamento")
            .agg(F.count("*").alias("n"), F.sum("valor_pen_valido").alias("monto_total"),
                 F.avg("valor_pen_valido").alias("monto_prom"))
            .orderBy(F.desc("n")).collect())

def agg1_rdd():
    # map -> (dep, (1, monto, 1 si hay monto)) ; reduceByKey suma componente a componente
    r = (base.rdd.map(lambda r: (r.entidad_departamento,
                                 (1, r.valor_pen_valido or 0.0, 1 if r.valor_pen_valido else 0)))
         .reduceByKey(lambda a,b: (a[0]+b[0], a[1]+b[1], a[2]+b[2]))
         .map(lambda kv: (kv[0], kv[1][0], kv[1][1], kv[1][1]/kv[1][2] if kv[1][2] else None))
         .sortBy(lambda x: -x[1]))
    return r.collect()

res_df, t_df_min, t_df_med = cronometrar(agg1_df)
res_rdd, t_rdd_min, t_rdd_med = cronometrar(agg1_rdd)
print(f"Agg1 DataFrame: min {t_df_min:.2f}s  mediana {t_df_med:.2f}s")
print(f"Agg1 RDD      : min {t_rdd_min:.2f}s  mediana {t_rdd_med:.2f}s")
print("Top-3 DF :", [(r[0], r[1]) for r in res_df[:3]])
print("Top-3 RDD:", [(r[0], r[1]) for r in res_rdd[:3]])

Agg1 DataFrame: min 1.24s  mediana 1.30s
Agg1 RDD      : min 3.44s  mediana 3.97s
Top-3 DF : [('LIMA', 62796), ('CUSCO', 17847), ('ANCASH', 11935)]
Top-3 RDD: [('LIMA', 62796), ('CUSCO', 17847), ('ANCASH', 11935)]


In [4]:
# ---------- Agg 2 ----------
def agg2_df():
    return (base.filter(~F.col("estado_proceso").isin("CONVOCADO","sin_dato"))
            .groupBy("procedimiento","categoria")
            .agg(F.count("*").alias("n"),
                 F.sum((F.col("estado_proceso")=="DESIERTO").cast("int")).alias("n_desierto"))
            .withColumn("tasa_desierto", F.col("n_desierto")/F.col("n"))
            .filter("n >= 200").orderBy(F.desc("tasa_desierto")).collect())

def agg2_rdd():
    r = (base.rdd.filter(lambda r: r.estado_proceso not in ("CONVOCADO","sin_dato"))
         .map(lambda r: ((r.procedimiento, r.categoria), (1, 1 if r.estado_proceso=="DESIERTO" else 0)))
         .reduceByKey(lambda a,b: (a[0]+b[0], a[1]+b[1]))
         .filter(lambda kv: kv[1][0] >= 200)
         .map(lambda kv: (kv[0][0], kv[0][1], kv[1][0], kv[1][1], kv[1][1]/kv[1][0]))
         .sortBy(lambda x: -x[4]))
    return r.collect()

r2_df, a, b = cronometrar(agg2_df); print(f"Agg2 DataFrame: min {a:.2f}s mediana {b:.2f}s")
r2_rdd, a, b = cronometrar(agg2_rdd); print(f"Agg2 RDD      : min {a:.2f}s mediana {b:.2f}s")
pd.DataFrame([tuple(r) for r in r2_df], columns=["procedimiento","categoria","n","n_desierto","tasa_desierto"]).head(10)

Agg2 DataFrame: min 1.61s mediana 2.10s
Agg2 RDD      : min 3.62s mediana 4.34s


,procedimiento,categoria,n,n_desierto,tasa_desierto
0,Adjudicación Selectiva,services,588,189,0.321429
1,Adjudicación Selectiva,goods,433,91,0.210162
2,Concurso Público Abreviado,services,24642,4682,0.190001
3,Licitación Pública Abreviada,goods,16429,2762,0.168117
4,Concurso Público para Consultoría,services,1288,210,0.163043
5,Subasta Inversa Electrónica,goods,23394,3700,0.158160
6,Adjudicación Simplificada,services,30535,4575,0.149828
7,Adjudicación Simplificada,goods,21056,2979,0.141480
8,Concurso Público de Servicios,services,3778,485,0.128375
9,Comparación de Precios,goods,12043,1437,0.119322


### Particionamiento
Los Parquet están particionados por `anio_pub`. Abajo se ve cuántas particiones usa Spark, qué hace `groupBy` (shuffle) y cómo cambia el tiempo al reparticionar.

In [5]:
print("Particiones de lectura (procesos):", procesos.rdd.getNumPartitions())
print("spark.sql.shuffle.partitions     :", spark.conf.get("spark.sql.shuffle.partitions"))

q = base.groupBy("entidad_departamento").count()
q.explain()          # busca 'Exchange hashpartitioning' = shuffle

def con_particiones(n, por=None):
    d = base.repartition(n, por) if por else base.repartition(n)
    t=time.perf_counter()
    d.groupBy("entidad_departamento").agg(F.count("*"), F.sum("valor_pen_valido")).collect()
    return time.perf_counter()-t

filas=[]
for n in [1,2,4,8,16,64,200]:
    ts=[con_particiones(n) for _ in range(3)]
    filas.append(("repartition(n)", n, min(ts)))
for n in [4,8,16]:
    ts=[con_particiones(n,"entidad_departamento") for _ in range(3)]
    filas.append(("repartition(n, dep)", n, min(ts)))
df_part = pd.DataFrame(filas, columns=["estrategia","n_part","t_min_s"]); display(df_part)

# sesgo: filas por partición al particionar por departamento (Lima domina)
sesgo = (base.repartition(8,"entidad_departamento").rdd.glom().map(len).collect()); print("filas/partición (por dep):", sesgo)
# partition pruning por anio_pub
procesos.filter("anio_pub = 2025").groupBy("periodo").count().explain()
df_part.to_csv(f"{OUT}/tarea4_particiones.csv", index=False)

Particiones de lectura (procesos): 3
spark.sql.shuffle.partitions     : 200
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[entidad_departamento#41], functions=[count(1)])
   +- Exchange hashpartitioning(entidad_departamento#41, 200), ENSURE_REQUIREMENTS, [plan_id=963]
      +- HashAggregate(keys=[entidad_departamento#41], functions=[partial_count(1)])
         +- InMemoryTableScan [entidad_departamento#41]
               +- InMemoryRelation [entidad_departamento#41, categoria#10, procedimiento#9, estado_proceso#30, valor_pen_valido#24, anio_pub#46], StorageLevel(disk, memory, deserialized, 1 replicas)
                     +- *(1) Project [entidad_departamento#41, categoria#10, procedimiento#9, estado_proceso#30, valor_pen_valido#24, anio_pub#46]
                        +- *(1) ColumnarToRow
                           +- FileScan parquet [procedimiento#9,categoria#10,valor_pen_valido#24,estado_proceso#30,entidad_departamento#41,anio_pub#46] Batched: true,

,estrategia,n_part,t_min_s
0,repartition(n),1,0.839007
1,repartition(n),2,0.854735
2,repartition(n),4,0.633297
3,repartition(n),8,0.719067
4,repartition(n),16,1.085252
5,repartition(n),64,2.019744
6,repartition(n),200,4.127919
7,"repartition(n, dep)",4,0.302918
8,"repartition(n, dep)",8,0.347613
9,"repartition(n, dep)",16,0.435215


filas/partición (por dep): [17757, 61690, 8834, 7164, 3976, 6845, 74055, 29419]
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[periodo#22], functions=[count(1)])
   +- Exchange hashpartitioning(periodo#22, 200), ENSURE_REQUIREMENTS, [plan_id=2952]
      +- HashAggregate(keys=[periodo#22], functions=[partial_count(1)])
         +- Project [periodo#22]
            +- FileScan parquet [periodo#22,anio_pub#46] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/content/drive/MyDrive/Compartidos/Proyecto Data Mining/proyecto_..., PartitionFilters: [isnotnull(anio_pub#46), (anio_pub#46 = 2025)], PushedFilters: [], ReadSchema: struct<periodo:string>




## 2. Tarea 16 — Definición de transacciones
**Criterio:** cada **proceso de contratación** (`release_id`) es una canasta. Ítems (con prefijo para evitar colisiones):

| Prefijo | Origen | Justificación |
|---|---|---|
| `cat=` | `categoria` | bienes/servicios/obras |
| `proc=` | `procedimiento` | régimen de contratación |
| `tipo=` | `entidad_tipo` | quién contrata (heurístico, 23% `otro`) |
| `dep=` | `entidad_departamento` | sede de la entidad, **no** lugar de ejecución |
| `seg=` | `items.unspsc_segmento` (varios por proceso) | rubro |
| `monto=` | cuartiles de `valor_pen_valido` (+`sin_dato`/`protegido`) | tamaño |
| `postores=` | `n_postores` en tramos | competencia |
| `ley=` | `fecha_pub` antes/después de 2025-04-22 | régimen legal |
| `estado=` | `estado_proceso` | resultado (el consecuente de interés) |

**Decisiones:** se excluyen `CONVOCADO` (aún abiertos) y estados `sin_dato`. Se agrupan los estados raros en `OTRO`. No se incluyen identificadores (RUC/entidad) para no producir reglas triviales de ítem único.

In [6]:
FECHA_LEY = "2025-04-22"
qs = procesos.filter("valor_pen_valido is not null").approxQuantile("valor_pen_valido",[0.25,0.5,0.75],0.001)
print("Cuartiles de monto (S/):", qs)

estados_ppales = ["CONTRATADO","DESIERTO","NULO","CONSENTIDO","ADJUDICADO","MIXTO","CANCELADO","RETROTRAIDO_POR_RESOLUCION"]

p = procesos.filter(~F.col("estado_proceso").isin("CONVOCADO","sin_dato")).select(
    "release_id","categoria","procedimiento","entidad_tipo","entidad_departamento",
    "estado_proceso","valor_pen_valido","info_protegida","n_postores","fecha_pub")

monto = (F.when(F.col("valor_pen_valido").isNull() & F.col("info_protegida"), "monto=protegido")
          .when(F.col("valor_pen_valido").isNull(), "monto=sin_dato")
          .when(F.col("valor_pen_valido")<=qs[0], "monto=Q1_bajo")
          .when(F.col("valor_pen_valido")<=qs[1], "monto=Q2")
          .when(F.col("valor_pen_valido")<=qs[2], "monto=Q3")
          .otherwise("monto=Q4_alto"))
post = (F.when(F.col("n_postores")==0,"postores=0").when(F.col("n_postores")==1,"postores=1")
         .when(F.col("n_postores")<=4,"postores=2-4").when(F.col("n_postores")<=9,"postores=5-9")
         .otherwise("postores=10+"))
estado = F.concat(F.lit("estado="), F.when(F.col("estado_proceso").isin(estados_ppales), F.col("estado_proceso")).otherwise("OTRO"))

p = p.withColumn("base_items", F.array(
        F.concat(F.lit("cat="), F.col("categoria")), F.concat(F.lit("proc="), F.col("procedimiento")),
        F.concat(F.lit("tipo="), F.col("entidad_tipo")), F.concat(F.lit("dep="), F.col("entidad_departamento")),
        monto, post,
        F.when(F.col("fecha_pub")>=F.lit(FECHA_LEY),"ley=nueva_32069").otherwise("ley=anterior"),
        estado))

segs = (items.filter("unspsc_segmento is not null")
        .groupBy("release_id").agg(F.collect_set(F.concat(F.lit("seg="), F.col("unspsc_segmento"))).alias("segs")))

trans = (p.join(segs,"release_id","left")
          .withColumn("items", F.array_distinct(F.concat("base_items", F.coalesce("segs", F.array()))))
          .select("release_id","items").cache())
N = trans.count()
print("Transacciones:", N)
trans.select(F.size("items").alias("k")).describe().show()
trans.show(3, truncate=False)

freq_items = (trans.select(F.explode("items").alias("item")).groupBy("item").count()
              .withColumn("soporte", F.col("count")/N).orderBy(F.desc("count")).toPandas())
print("Ítems distintos:", len(freq_items)); display(freq_items.head(25))

Cuartiles de monto (S/): [90360.0, 195000.0, 506325.85]
Transacciones: 196158
+-------+------------------+
|summary|                 k|
+-------+------------------+
|  count|            196158|
|   mean|  8.84385546345293|
| stddev|0.3902154225955744|
|    min|                 8|
|    max|                18|
+-------+------------------+

+------------------------------------------------------------+---------------------------------------------------------------------------------------------------------------------------------------+
|release_id                                                  |items                                                                                                                                  |
+------------------------------------------------------------+---------------------------------------------------------------------------------------------------------------------------------------+
|ocds-dgv273-seacev3-1081886-2025-01-18T10:02:44.892791-05:00|[

,item,count,soporte
0,estado=CONTRATADO,131154,0.668614
1,ley=anterior,98837,0.503864
2,ley=nueva_32069,97321,0.496136
3,postores=10+,87932,0.448271
4,cat=services,86591,0.441435
5,cat=goods,86088,0.438871
6,tipo=municipalidad,81493,0.415446
7,proc=Adjudicación Simplificada,59098,0.301278
8,dep=LIMA,58823,0.299876
9,seg=72,46648,0.237808


## 3. Tarea 17 — Apriori (sobre una muestra)
Implementación propia (generación de candidatos por join + poda por antimonotonía, conteo con matriz booleana numpy). Se corre sobre una muestra para poder compararlo luego con FP-Growth sobre **las mismas** transacciones.

In [7]:
MUESTRA_N = 30000
muestra = trans.sample(False, min(1.0, MUESTRA_N/N), seed=42).toPandas()
T = [set(x) for x in muestra["items"]]; n = len(T); print("Muestra:", n)

def apriori(T, min_sup, max_k=5):
    n=len(T); cnt={}
    for t in T:
        for i in t: cnt[i]=cnt.get(i,0)+1
    L1={i:c for i,c in cnt.items() if c/n>=min_sup}
    vocab=sorted(L1); idx={v:k for k,v in enumerate(vocab)}
    X=np.zeros((n,len(vocab)),dtype=bool)
    for r,t in enumerate(T):
        for i in t:
            if i in idx: X[r,idx[i]]=True
    res={(v,):c for v,c in L1.items()}
    prev=[(v,) for v in vocab]; k=2
    while prev and k<=max_k:
        prevset=set(prev); cands=set()
        for a,b in itertools.combinations(prev,2):
            if a[:-1]==b[:-1]:
                c=a+(b[-1],)
                if all(c[:j]+c[j+1:] in prevset for j in range(len(c))):   # poda
                    cands.add(c)
        nxt=[]
        for c in cands:
            s=int(X[:,[idx[i] for i in c]].all(axis=1).sum())
            if s/n>=min_sup: res[c]=s; nxt.append(c)
        print(f"  k={k}: candidatos {len(cands)} -> frecuentes {len(nxt)}")
        prev=sorted(nxt); k+=1
    return res

MIN_SUP = 0.02
_, t_ap, _ = cronometrar(lambda: apriori(T, MIN_SUP), n=1)
fi_apriori = apriori(T, MIN_SUP)
print(f"Apriori (muestra {n}, minsup={MIN_SUP}): {len(fi_apriori)} itemsets, {t_ap:.1f}s")

Muestra: 30188
  k=2: candidatos 2016 -> frecuentes 359
  k=3: candidatos 1351 -> frecuentes 738
  k=4: candidatos 958 -> frecuentes 607
  k=5: candidatos 266 -> frecuentes 201
  k=2: candidatos 2016 -> frecuentes 359
  k=3: candidatos 1351 -> frecuentes 738
  k=4: candidatos 958 -> frecuentes 607
  k=5: candidatos 266 -> frecuentes 201
Apriori (muestra 30188, minsup=0.02): 1969 itemsets, 1.7s


## 4. Tarea 18 — FP-Growth (Spark MLlib)
Primero sobre **la misma muestra** (para validar que ambos algoritmos dan los mismos itemsets), luego sobre **todo el dataset**.

In [8]:
from pyspark.ml.fpm import FPGrowth

def fpg(df, min_sup):
    t=time.perf_counter()
    m = FPGrowth(itemsCol="items", minSupport=min_sup, minConfidence=0.0).fit(df)
    fi = {tuple(sorted(r["items"])): r["freq"] for r in m.freqItemsets.collect()}
    return fi, time.perf_counter()-t

# (a) misma muestra que Apriori
ms = spark.createDataFrame(muestra[["release_id","items"]])
fi_fpg_m, t_fpg_m = fpg(ms, MIN_SUP)
a = {tuple(sorted(k)):v for k,v in fi_apriori.items()}
print(f"Muestra: Apriori {t_ap:.1f}s ({len(a)}) | FP-Growth {t_fpg_m:.1f}s ({len(fi_fpg_m)})")
print("¿Mismos itemsets y mismos conteos?", a == fi_fpg_m)

# (b) dataset completo
MIN_SUP_FULL = 0.01
fi_full, t_fpg_full = fpg(trans, MIN_SUP_FULL)
print(f"FP-Growth COMPLETO ({N} transacciones, minsup={MIN_SUP_FULL}): {len(fi_full)} itemsets, {t_fpg_full:.1f}s")
print(pd.Series([len(k) for k in fi_full]).value_counts().sort_index())

# curva tiempo / nº itemsets vs soporte (completo)
filas=[]
for s in [0.05,0.02,0.01,0.005]:
    fi,t = fpg(trans,s); filas.append((s,len(fi),t))
curva = pd.DataFrame(filas,columns=["min_sup","n_itemsets","t_s"]); display(curva)
curva.to_csv(f"{OUT}/curva_fpgrowth.csv",index=False)

Muestra: Apriori 1.7s (1969) | FP-Growth 7.7s (2005)
¿Mismos itemsets y mismos conteos? False
FP-Growth COMPLETO (196158 transacciones, minsup=0.01): 5002 itemsets, 15.6s
1      84
2     637
3    1469
4    1630
5     877
6     239
7      54
8      11
9       1
Name: count, dtype: int64


,min_sup,n_itemsets,t_s
0,0.050,472,5.822982
1,0.020,1973,14.469802
2,0.010,5002,17.056696
3,0.005,11351,18.321572


## 5. Tarea 19 — Reglas (soporte, confianza, lift)
Para cada itemset frecuente \(I\) y partición \(A \Rightarrow C\):
\(\text{conf}=\text{sup}(I)/\text{sup}(A)\), \(\text{lift}=\text{conf}/\text{sup}(C)\). Por antimonotonía todos los subconjuntos están en los itemsets frecuentes, así que no hace falta volver a leer los datos.

Para reglas **no triviales** se descartan las que mezclan ítems de la misma variable (p. ej. dos `proc=`) y las que son casi deterministas por construcción (`ley=` ⇒ procedimientos nuevos, y viceversa: se marcan y se reportan aparte).

In [9]:
def reglas(fi, N, min_conf=0.3, min_lift=1.0):
    sup = {frozenset(k): v/N for k,v in fi.items()}
    out=[]
    for I,s in sup.items():
        if len(I)<2: continue
        for r in range(1,len(I)):
            for A in itertools.combinations(sorted(I), r):
                A=frozenset(A); Cn=I-A
                conf=s/sup[A]; lift=conf/sup[Cn]
                if conf>=min_conf and lift>=min_lift:
                    out.append((" & ".join(sorted(A))," & ".join(sorted(Cn)),s,conf,lift,len(A),len(Cn)))
    return pd.DataFrame(out,columns=["antecedente","consecuente","soporte","confianza","lift","k_ant","k_cons"])

R = reglas(fi_full, N, min_conf=0.3, min_lift=1.0)
pref = lambda s: {x.split("=")[0] for x in s.split(" & ")}
R["trivial_ley_proc"] = [(("ley" in pref(a)) and ("proc" in pref(c))) or (("proc" in pref(a)) and ("ley" in pref(c)))
                          for a,c in zip(R.antecedente,R.consecuente)]
R["mismo_tipo"] = [len(pref(a)&pref(c))>0 for a,c in zip(R.antecedente,R.consecuente)]
R = R[~R.mismo_tipo].copy()
print("Reglas:", len(R), "| triviales ley/proc:", R.trivial_ley_proc.sum())
R.sort_values("lift",ascending=False).head(15)

Reglas: 33804 | triviales ley/proc: 9110


,antecedente,consecuente,soporte,confianza,lift,k_ant,k_cons,trivial_ley_proc,mismo_tipo
13103,dep=LIMA & estado=CONTRATADO & monto=Q1_bajo &...,cat=services & ley=anterior & proc=Regímen Esp...,0.010043,0.615048,53.836083,5,4,False,False
12949,cat=services & ley=anterior & proc=Regímen Esp...,dep=LIMA & estado=CONTRATADO & monto=Q1_bajo &...,0.010043,0.879072,53.836083,4,5,False,False
12898,estado=CONTRATADO & proc=Regímen Especial & se...,cat=services & dep=LIMA & ley=anterior & monto...,0.010043,0.890194,53.828217,3,6,True,False
13170,cat=services & dep=LIMA & ley=anterior & monto...,estado=CONTRATADO & proc=Regímen Especial & se...,0.010043,0.607275,53.828217,6,3,True,False
12982,dep=LIMA & monto=Q1_bajo & postores=1 & seg=80,cat=services & estado=CONTRATADO & ley=anterio...,0.010043,0.553526,53.250910,4,5,False,False
13065,cat=services & estado=CONTRATADO & ley=anterio...,dep=LIMA & monto=Q1_bajo & postores=1 & seg=80,0.010043,0.966160,53.250910,5,4,False,False
13079,cat=services & ley=anterior & monto=Q1_bajo & ...,dep=LIMA & estado=CONTRATADO & proc=Regímen Es...,0.010043,0.596067,53.002369,5,4,True,False
12968,dep=LIMA & estado=CONTRATADO & proc=Regímen Es...,cat=services & ley=anterior & monto=Q1_bajo & ...,0.010043,0.893019,53.002369,4,5,True,False
13608,ley=anterior & proc=Regímen Especial & tipo=otro,dep=LIMA & estado=CONTRATADO & monto=Q1_bajo &...,0.010043,0.863278,52.868827,3,5,False,False
12908,ley=anterior & proc=Regímen Especial & tipo=otro,cat=services & dep=LIMA & estado=CONTRATADO & ...,0.010043,0.863278,52.868827,3,6,False,False


In [10]:
RN = R[~R.trivial_ley_proc]
cols=["antecedente","consecuente","soporte","confianza","lift"]
print("== Consecuente = estado DESIERTO =="); display(RN[RN.consecuente=="estado=DESIERTO"].sort_values("lift",ascending=False)[cols].head(10))
print("== Consecuente = estado NULO ==");     display(RN[RN.consecuente=="estado=NULO"].sort_values("lift",ascending=False)[cols].head(10))
print("== Mayor lift con soporte >= 2% ==");   display(RN[RN.soporte>=0.02].sort_values("lift",ascending=False)[cols].head(15))
print("== Reglas con ley=nueva_32069 en el antecedente =="); display(RN[RN.antecedente.str.contains("ley=")].sort_values("lift",ascending=False)[cols].head(10))

== Consecuente = estado DESIERTO ==


,antecedente,consecuente,soporte,confianza,lift
27642,monto=protegido & postores=0,estado=DESIERTO,0.010864,0.386611,3.001300
27649,ley=anterior & monto=protegido & postores=0,estado=DESIERTO,0.010221,0.379663,2.947361
27763,postores=0,estado=DESIERTO,0.015987,0.354511,2.752100
27765,ley=anterior & postores=0,estado=DESIERTO,0.012918,0.320111,2.485055
27685,monto=protegido,estado=DESIERTO,0.012108,0.311190,2.415797
27687,ley=anterior & monto=protegido,estado=DESIERTO,0.011404,0.306817,2.381848


== Consecuente = estado NULO ==


,antecedente,consecuente,soporte,confianza,lift
27795,ley=anterior & postores=0 & tipo=municipalidad,estado=NULO,0.011960,0.622942,10.290979
27787,postores=0 & tipo=municipalidad,estado=NULO,0.012602,0.595089,9.830847
27784,ley=anterior & postores=0,estado=NULO,0.020463,0.507074,8.376847
27779,ley=anterior & postores=0 & proc=Adjudicación ...,estado=NULO,0.012959,0.491778,8.124151
27770,postores=0 & proc=Adjudicación Simplificada,estado=NULO,0.013275,0.485187,8.015274
27767,postores=0,estado=NULO,0.021518,0.477165,7.882744
27660,ley=anterior & monto=protegido & postores=0,estado=NULO,0.010853,0.403143,6.659912
27652,monto=protegido & postores=0,estado=NULO,0.011073,0.394049,6.509679


== Mayor lift con soporte >= 2% ==


,antecedente,consecuente,soporte,confianza,lift
27635,monto=protegido,ley=anterior & postores=0,0.026922,0.691955,17.146601
27638,ley=anterior & postores=0,monto=protegido,0.026922,0.667130,17.146601
27636,postores=0,ley=anterior & monto=protegido,0.026922,0.596993,16.061576
27637,ley=anterior & monto=protegido,postores=0,0.026922,0.724318,16.061576
27591,monto=protegido,postores=0,0.028100,0.722222,16.015110
27592,postores=0,monto=protegido,0.028100,0.623106,16.015110
27781,postores=0,estado=NULO & ley=anterior,0.020463,0.453764,14.717183
27782,estado=NULO & ley=anterior,postores=0,0.020463,0.663690,14.717183
30606,cat=goods & postores=2-4 & proc=Subasta Invers...,seg=15,0.021610,0.590472,14.693118
30601,seg=15,cat=goods & postores=2-4 & proc=Subasta Invers...,0.021610,0.537739,14.693118


== Reglas con ley=nueva_32069 en el antecedente ==


,antecedente,consecuente,soporte,confianza,lift
12949,cat=services & ley=anterior & proc=Regímen Esp...,dep=LIMA & estado=CONTRATADO & monto=Q1_bajo &...,0.010043,0.879072,53.836083
13065,cat=services & estado=CONTRATADO & ley=anterio...,dep=LIMA & monto=Q1_bajo & postores=1 & seg=80,0.010043,0.966160,53.250910
12908,ley=anterior & proc=Regímen Especial & tipo=otro,cat=services & dep=LIMA & estado=CONTRATADO & ...,0.010043,0.863278,52.868827
13608,ley=anterior & proc=Regímen Especial & tipo=otro,dep=LIMA & estado=CONTRATADO & monto=Q1_bajo &...,0.010043,0.863278,52.868827
12999,estado=CONTRATADO & ley=anterior & proc=Regíme...,cat=services & dep=LIMA & monto=Q1_bajo & post...,0.010043,0.953533,52.554989
13657,estado=CONTRATADO & ley=anterior & proc=Regíme...,dep=LIMA & monto=Q1_bajo & postores=1 & seg=80,0.010043,0.953533,52.554989
12722,cat=services & ley=anterior & proc=Regímen Esp...,dep=LIMA & monto=Q1_bajo & postores=1 & seg=80,0.010889,0.953146,52.533632
12932,cat=services & estado=CONTRATADO & ley=anterio...,dep=LIMA & monto=Q1_bajo & postores=1 & seg=80...,0.010043,0.853183,51.765762
13496,ley=anterior & proc=Regímen Especial & tipo=otro,dep=LIMA & monto=Q1_bajo & postores=1 & seg=80,0.010889,0.936021,51.589776
12691,ley=anterior & proc=Regímen Especial & tipo=otro,cat=services & dep=LIMA & monto=Q1_bajo & post...,0.010889,0.936021,51.589776


### Selección y lectura de las ≥ 8 reglas
Ejecuta la celda anterior y **pega la salida en el chat**: con los valores reales redacto la interpretación (contexto SEACE, Ley 32069, sesgo de sede de entidad, etc.). La celda siguiente arma una tabla candidata balanceada (DESIERTO, NULO, otros, ley) y valida con un baseline.

In [11]:
def top(df,n,key="lift"): return df.sort_values(key,ascending=False).head(n)
cand = pd.concat([
    top(RN[(RN.consecuente=="estado=DESIERTO")&(RN.lift>1.2)],4),
    top(RN[(RN.consecuente=="estado=NULO")&(RN.lift>1.2)],2),
    top(RN[(RN.soporte>=0.02)&(~RN.consecuente.str.startswith("estado="))],4),
    top(RN[RN.antecedente.str.contains("ley=")],2),
]).drop_duplicates(["antecedente","consecuente"])
# baseline: soporte marginal del consecuente (lift=1 significa independencia)
supm = {k:v/N for k,v in fi_full.items() if len(k)==1}
cand["sup_consecuente"] = [supm.get((c,),np.nan) for c in cand.consecuente]
cand.round(4)

,antecedente,consecuente,soporte,confianza,lift,k_ant,k_cons,trivial_ley_proc,mismo_tipo,sup_consecuente
27642,monto=protegido & postores=0,estado=DESIERTO,0.0109,0.3866,3.0013,2,1,False,False,0.1288
27649,ley=anterior & monto=protegido & postores=0,estado=DESIERTO,0.0102,0.3797,2.9474,3,1,False,False,0.1288
27763,postores=0,estado=DESIERTO,0.0160,0.3545,2.7521,1,1,False,False,0.1288
27765,ley=anterior & postores=0,estado=DESIERTO,0.0129,0.3201,2.4851,2,1,False,False,0.1288
27795,ley=anterior & postores=0 & tipo=municipalidad,estado=NULO,0.0120,0.6229,10.2910,3,1,False,False,0.0605
27787,postores=0 & tipo=municipalidad,estado=NULO,0.0126,0.5951,9.8308,2,1,False,False,0.0605
27635,monto=protegido,ley=anterior & postores=0,0.0269,0.6920,17.1466,1,2,False,False,NaN
27638,ley=anterior & postores=0,monto=protegido,0.0269,0.6671,17.1466,2,1,False,False,0.0389
27637,ley=anterior & monto=protegido,postores=0,0.0269,0.7243,16.0616,2,1,False,False,0.0451
27636,postores=0,ley=anterior & monto=protegido,0.0269,0.5970,16.0616,1,2,False,False,NaN


## 6. Export de artefactos para la web (módulo interactivo: umbrales de soporte/confianza/lift)

In [12]:
R_out = R.copy()
R_out["soporte"]=R_out.soporte.round(5); R_out["confianza"]=R_out.confianza.round(4); R_out["lift"]=R_out.lift.round(3)
R_out.to_csv(f"{OUT}/reglas_asociacion.csv", index=False)
R_out.to_json(f"{OUT}/reglas_asociacion.json", orient="records", force_ascii=False)
pd.DataFrame([(" & ".join(k),v,v/N) for k,v in fi_full.items()],columns=["itemset","freq","soporte"]).to_csv(f"{OUT}/itemsets_frecuentes.csv",index=False)
freq_items.to_csv(f"{OUT}/items_frecuencia.csv",index=False)
cand.to_csv(f"{OUT}/reglas_seleccionadas.csv",index=False)
pd.DataFrame([("apriori_muestra",n,MIN_SUP,len(a),t_ap),("fpgrowth_muestra",n,MIN_SUP,len(fi_fpg_m),t_fpg_m),
              ("fpgrowth_completo",N,MIN_SUP_FULL,len(fi_full),t_fpg_full)],
             columns=["algoritmo","n_trans","min_sup","n_itemsets","t_s"]).to_csv(f"{OUT}/comparacion_apriori_fpgrowth.csv",index=False)
print(os.listdir(OUT))

['tarea4_particiones.csv', 'curva_fpgrowth.csv', 'reglas_asociacion.csv', 'reglas_asociacion.json', 'itemsets_frecuentes.csv', 'items_frecuencia.csv', 'reglas_seleccionadas.csv', 'comparacion_apriori_fpgrowth.csv']
